# Business Forecasting — Exponential Smoothing Models
### Simple exponential smoothing, Holt trend, damped trend, Holt–Winters, and ETS thinking

This notebook is designed to work in **Google Colab** and also locally. It follows the logic of Chapter 8 of *Forecasting: Principles and Practice — the Pythonic Way*, while using `statsmodels` so students can see each method directly.

**Learning goals**
1. Match a smoothing method to the structure of a time series.
2. Explain the meaning of level, trend, seasonality, and smoothing parameters.
3. Fit SES, Holt, damped Holt, and Holt–Winters models in Python.
4. Compare forecasts on a holdout set using MAE, RMSE, and MAPE.
5. Explain why additive and multiplicative seasonality lead to different forecasts.

> The main question throughout the notebook is not “Which function do I call?” It is: **What structure is present in the data, and which model represents that structure?**

**Data note:** The Nile and U.S. macro examples are loaded from the `statsmodels` teaching datasets; the AirPassengers values are embedded below. The notebook does not fetch a dataset from the internet. Run the checkpoint questions yourself before moving to the wrap-up.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import math
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import statsmodels.api as sm
from statsmodels.tsa.holtwinters import SimpleExpSmoothing, Holt, ExponentialSmoothing
from sklearn.metrics import mean_absolute_error, mean_squared_error

pd.set_option('display.precision', 3)
plt.rcParams['figure.figsize'] = (10, 5)

def accuracy(actual, forecast):
    actual = np.asarray(actual, dtype=float)
    forecast = np.asarray(forecast, dtype=float)
    mae = mean_absolute_error(actual, forecast)
    rmse = math.sqrt(mean_squared_error(actual, forecast))
    mape = np.mean(np.abs((actual - forecast) / actual)) * 100
    return pd.Series({'MAE': mae, 'RMSE': rmse, 'MAPE (%)': mape})

## 1. Load the three datasets
We deliberately use three different shapes so the model choice becomes visible.

In [ ]:
# Dataset 1: Nile annual river flow, bundled with statsmodels.
nile = sm.datasets.nile.load_pandas().data.copy()
nile['year'] = nile['year'].astype(int)
nile = nile.rename(columns={'volume':'y'})

# Dataset 2: quarterly U.S. macro data. We keep Q4 population as one annual observation.
macro = sm.datasets.macrodata.load_pandas().data.copy()
pop = macro.loc[macro['quarter'] == 4, ['year','pop']].copy()
pop['year'] = pop['year'].astype(int)
pop = pop.rename(columns={'pop':'y'})

# Dataset 3: classic monthly AirPassengers data, embedded so this notebook does not need internet access.
air_values = [
112,118,132,129,121,135,148,148,136,119,104,118,
115,126,141,135,125,149,170,170,158,133,114,140,
145,150,178,163,172,178,199,199,184,162,146,166,
171,180,193,181,183,218,230,242,209,191,172,194,
196,196,236,235,229,243,264,272,237,211,180,201,
204,188,235,227,234,264,302,293,259,229,203,229,
242,233,267,269,270,315,364,347,312,274,237,278,
284,277,317,313,318,374,413,405,355,306,271,306,
315,301,356,348,355,422,465,467,404,347,305,336,
340,318,362,348,363,435,491,505,404,359,310,337,
360,342,406,396,420,472,548,559,463,407,362,405,
417,391,419,461,472,535,622,606,508,461,390,432]
air_dates = pd.date_range('1949-01-01', periods=len(air_values), freq='MS')
air = pd.DataFrame({'date': air_dates, 'y': air_values})

print('Nile rows:', len(nile), '| Population rows:', len(pop), '| AirPassengers rows:', len(air))

## 2. Start with the pattern, not the model
Plot all three series. Before fitting anything, classify each one in terms of **level, trend, and seasonality**.

In [ ]:
fig, axes = plt.subplots(3, 1, figsize=(11, 11))
axes[0].plot(nile['year'], nile['y']); axes[0].set_title('Nile: annual river flow'); axes[0].set_ylabel('Volume')
axes[1].plot(pop['year'], pop['y']); axes[1].set_title('U.S. population: annual Q4 values'); axes[1].set_ylabel('Millions')
axes[2].plot(air['date'], air['y']); axes[2].set_title('AirPassengers: monthly'); axes[2].set_ylabel('Passengers')
plt.tight_layout(); plt.show()

## 3. Simple Exponential Smoothing (SES)
SES has one evolving component: the **level**.

\[
\ell_t = \alpha y_t + (1-\alpha)\ell_{t-1}
\]
\[
\hat{y}_{t+h|t}=\ell_t
\]

The multi-step forecast is flat because SES has no trend or seasonal component.

In [ ]:
nile_s = nile.set_index('year')['y'].astype(float)
train_nile, test_nile = nile_s.iloc[:-10], nile_s.iloc[-10:]

ses = SimpleExpSmoothing(train_nile, initialization_method='estimated').fit(optimized=True)
ses_fc = ses.forecast(len(test_nile))
print('Estimated alpha:', round(ses.params['smoothing_level'], 4))
print(accuracy(test_nile, ses_fc))

plt.plot(train_nile.index[-40:], train_nile.iloc[-40:], label='Train')
plt.plot(test_nile.index, test_nile, label='Test')
plt.plot(test_nile.index, ses_fc, label='SES forecast')
plt.legend(); plt.title('SES holdout forecast'); plt.show()

## 4. What does alpha actually do?
A larger \(alpha\) reacts more strongly to the latest observation. A smaller \(alpha\) produces a smoother level.

In [ ]:
plt.plot(nile_s.index, nile_s.values, label='Observed', linewidth=2)
for alpha in [0.2, 0.5, 0.8]:
    fit = SimpleExpSmoothing(nile_s, initialization_method='estimated').fit(
        smoothing_level=alpha, optimized=False)
    plt.plot(nile_s.index, fit.fittedvalues, label=f'alpha={alpha}')
plt.legend(); plt.title('Same data, different alpha values'); plt.show()

## 5. Holt's linear trend method
Holt adds a second evolving component: the trend \(b_t\).

\[
\ell_t=\alpha y_t+(1-\alpha)(\ell_{t-1}+b_{t-1})
\]
\[
b_t=\beta(\ell_t-\ell_{t-1})+(1-\beta)b_{t-1}
\]
\[
\hat{y}_{t+h|t}=\ell_t+h b_t
\]

In [ ]:
pop_s = pop.set_index('year')['y'].astype(float)
train_pop, test_pop = pop_s.iloc[:-10], pop_s.iloc[-10:]

holt = Holt(train_pop, damped_trend=False, initialization_method='estimated').fit(optimized=True)
holt_fc = holt.forecast(len(test_pop))
print('alpha:', round(holt.params['smoothing_level'], 4))
print('beta :', round(holt.params['smoothing_trend'], 4))
print(accuracy(test_pop, holt_fc))

plt.plot(train_pop.index[-30:], train_pop.iloc[-30:], label='Train')
plt.plot(test_pop.index, test_pop, label='Test')
plt.plot(test_pop.index, holt_fc, label='Holt forecast')
plt.legend(); plt.title("Holt's linear trend method"); plt.show()

## 6. Damped trend
A straight-line trend can become unrealistic at long horizons. Damping multiplies future trend contributions by powers of \(\phi\), where \(0<\phi<1\).

The result: short-run forecasts can trend, while the long-run path gradually flattens.

In [ ]:
damped = Holt(train_pop, damped_trend=True, initialization_method='estimated').fit(optimized=True)
damped_fc = damped.forecast(len(test_pop))

comparison = pd.DataFrame({
    'Actual': test_pop,
    'Holt': holt_fc,
    'Damped Holt': damped_fc
})
print(comparison.head())
print('\nAccuracy comparison:')
print(pd.DataFrame({
    'Holt': accuracy(test_pop, holt_fc),
    'Damped Holt': accuracy(test_pop, damped_fc)
}).T)

plt.plot(train_pop.index[-30:], train_pop.iloc[-30:], label='Train')
plt.plot(test_pop.index, test_pop, label='Test')
plt.plot(test_pop.index, holt_fc, label='Holt')
plt.plot(test_pop.index, damped_fc, label='Damped Holt')
plt.legend(); plt.title('Holt vs damped Holt'); plt.show()

## 7. Holt–Winters: trend + seasonality
For monthly data, the seasonal period is \(m=12\). Holt–Winters updates **level, trend, and seasonality**.

- **Additive seasonality:** seasonal swings are roughly constant in absolute size.
- **Multiplicative seasonality:** seasonal swings grow or shrink in proportion to the level.

In [ ]:
ap = air.set_index('date')['y'].astype(float)
train_ap, test_ap = ap.iloc[:-24], ap.iloc[-24:]

hw_add = ExponentialSmoothing(
    train_ap, trend='add', seasonal='add', seasonal_periods=12,
    initialization_method='estimated').fit(optimized=True)

hw_mul = ExponentialSmoothing(
    train_ap, trend='add', seasonal='mul', seasonal_periods=12,
    initialization_method='estimated').fit(optimized=True)

add_fc = hw_add.forecast(24)
mul_fc = hw_mul.forecast(24)

print(pd.DataFrame({
    'HW additive': accuracy(test_ap, add_fc),
    'HW multiplicative': accuracy(test_ap, mul_fc)
}).T)

plt.plot(train_ap.index[-60:], train_ap.iloc[-60:], label='Train')
plt.plot(test_ap.index, test_ap, label='Test')
plt.plot(test_ap.index, add_fc, label='Additive seasonality')
plt.plot(test_ap.index, mul_fc, label='Multiplicative seasonality')
plt.legend(); plt.title('Holt-Winters comparison'); plt.show()

## 8. Produce a 24-month business forecast
Once the model family is selected, refit it to all available data and forecast the required horizon.

In [ ]:
final_model = ExponentialSmoothing(
    ap,
    trend='add',
    damped_trend=True,
    seasonal='mul',
    seasonal_periods=12,
    initialization_method='estimated'
).fit(optimized=True)

future_24 = final_model.forecast(24)
print(future_24.head(12))

plt.plot(ap.index[-72:], ap.iloc[-72:], label='Observed')
plt.plot(future_24.index, future_24, label='24-month forecast')
plt.axvline(ap.index[-1], linestyle='--', linewidth=1)
plt.legend(); plt.title('Final 24-month Holt-Winters forecast'); plt.show()

## 9. ETS notation: a compact language
ETS describes three pieces: **Error, Trend, Seasonal**.

Examples:
- ETS(A,N,N): additive error, no trend, no seasonality — an SES-type model.
- ETS(A,A,N): additive error, additive trend, no seasonality — a Holt-type model.
- ETS(A,Ad,M): additive error, damped additive trend, multiplicative seasonality.

This notation helps us think about the model as a statistical system, not merely a smoothing recipe.

In [ ]:
# statsmodels does not use the ETS(A,A,M) string directly in the Holt-Winters class.
# Instead, we specify the components explicitly.
example = ExponentialSmoothing(
    ap,
    trend='add',
    damped_trend=True,
    seasonal='mul',
    seasonal_periods=12,
    initialization_method='estimated'
).fit()
print('Model fitted with: additive trend + damped trend + multiplicative seasonality')
print('Smoothing parameters:')
for k in ['smoothing_level','smoothing_trend','smoothing_seasonal','damping_trend']:
    print(k, ':', example.params.get(k))

## 10. Mini-checkpoint
For each situation, choose the most defensible starting model.

1. Weekly demand with no visible trend or seasonality.
2. Annual population with a steady upward trend.
3. Monthly sales with constant seasonal swings of about ±200 units.
4. Monthly sales where seasonal swings double when the average level doubles.
5. A growing product category where long-run linear growth seems unrealistic.

## 11. Wrap-up
Keep one rule in mind: **match the model to the pattern**. The Python function is the easy part; the forecasting judgment is deciding which components the data require.